# TissueMC: GPU rungs R5 and R6 on a Colab T4

The diagnostic run put R3's extra waiting on per-thread local memory. **R5** (one photon per thread) and **R6** (persistent lanes that relaunch) keep the random generator's buffer and the per-layer accumulator in registers instead of run-time-indexed arrays. With R1 and R3 they form a two-by-two comparison: arrays against registers, one photon per thread against relaunch.

This notebook checks R5 and R6 against the CPU on identical photons and validates them, benchmarks **R1, R3, R5, R6, then R1 again in the same session** (so the GPU's own drift is visible), and then collects every Nsight Compute counter for R3 and R6 and prints the ones that bear on local memory side by side. It ends with **a summary block and a digest** for you to paste back.

**Before you start:** *Runtime → Change runtime type → T4 GPU.* Nothing else to record.

**Step 1 is a short check.** Every later step starts only if it passed; otherwise it prints "SKIPPED", so *Run all* cannot waste time on a broken run. If step 1 fails, **paste its output**. Raw counters are kept in `results/*.ncu.csv` and reused if you run a step again. Expect about 25 minutes in all. Nothing here needs passwords or keys.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,clocks.max.sm --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
# Safe to run twice: reuse the clone if it is already there instead of nesting a second one.
!if [ -d /content/tissuemc/.git ]; then git -C /content/tissuemc pull -q; else git clone -q https://github.com/Raghava-Dwivedi/tissuemc.git /content/tissuemc; fi
%cd /content/tissuemc
!git log -1 --format='commit %h  %s'

In [ ]:
!pip install -q cmake ninja numpy matplotlib
!pip install -q -e .

In [ ]:
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DTISSUEMC_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 > /dev/null
!cmake --build build 2>&1 | tail -3
!./build/tissuemc --version

In [ ]:
# 1. Quick check: do R5 and R6 agree with the CPU on identical photons? The later steps run only if
# this passes (it leaves results/.quick_ok).
!rm -f results/.quick_ok
!tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 5 --photons 200000 && tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 6 --photons 200000 && touch results/.quick_ok

In [ ]:
# 2. Validate rungs 5 and 6 against the published values and the independent oracle.
!if [ -f results/.quick_ok ]; then tissuemc-validate --binary build/tissuemc --engine cuda --rung 5 --report results/gpu-r5-validation.json 2>&1 | grep -v '^  note' | tail -30; else echo "SKIPPED: the quick check did not pass, so validation of rung 5 was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-validate --binary build/tissuemc --engine cuda --rung 6 --report results/gpu-r6-validation.json 2>&1 | grep -v '^  note' | tail -30; else echo "SKIPPED: the quick check did not pass, so validation of rung 6 was not run"; fi

In [ ]:
# 3. Same photons through the CPU (float32) and each rung, on R_d, T_t and absorbed weight.
!if [ -f results/.quick_ok ]; then tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 5 --photons 2000000 --report results/gpu-r5-cross-engine.json; else echo "SKIPPED: the quick check did not pass, so the rung 5 comparison was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 6 --photons 2000000 --report results/gpu-r6-cross-engine.json; else echo "SKIPPED: the quick check did not pass, so the rung 6 comparison was not run"; fi

In [ ]:
# 4. Benchmarks back to back: R1, R3, R5, R6, then R1 again to show session drift. About 7 minutes.
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 1 --binary build/tissuemc --output results/gpu-r1-session.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the rung 1 benchmark was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 3 --binary build/tissuemc --output results/gpu-r3-session.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the rung 3 benchmark was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 5 --binary build/tissuemc --output results/gpu-r5.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the rung 5 benchmark was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 6 --binary build/tissuemc --output results/gpu-r6.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the rung 6 benchmark was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 1 --binary build/tissuemc --output results/gpu-r1-again.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the rung 1 benchmark was not run"; fi

In [ ]:
# 5. Every Nsight Compute counter for R3 and R6 (500,000 photons). About 3 to 6 minutes each.
!if [ -f results/.quick_ok ]; then tissuemc-profile --rung 3 --full --reuse-csv --photons 500000 --binary build/tissuemc --output results/gpu-r3-full.json; else echo "SKIPPED: the quick check did not pass, so the rung 3 profile was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-profile --rung 6 --full --reuse-csv --photons 500000 --binary build/tissuemc --output results/gpu-r6-full.json; else echo "SKIPPED: the quick check did not pass, so the rung 6 profile was not run"; fi

In [ ]:
# 6. The digest: A is R3, B is R6. Paste this cell's output back.
!if [ -f results/gpu-r3-full.json ] && [ -f results/gpu-r6-full.json ]; then tissuemc-profile --digest results/gpu-r3-full.json results/gpu-r6-full.json; else echo "SKIPPED: a profile above did not finish"; fi

In [ ]:
import sys

sys.path.insert(0, "python")
import json
import pathlib
import subprocess


def load(name):
    p = pathlib.Path("results") / name
    return json.loads(p.read_text()) if p.exists() else None


def bench_summary(b):
    if not b:
        return None
    return {
        "median_kernel_s": round(b["kernel"]["median"], 3),
        "kernel_seconds": [round(x, 3) for x in b["kernel_seconds"]],
        "million_events_per_s": round(b["events_per_second"] / 1e6, 1),
        "validated_first": b["validated_before_measuring"],
    }


def cross_summary(c):
    if not c:
        return None
    return {
        "agree": c["agree"],
        "gaps": {g["name"]: [g["value"], g["limit"]] for g in c["gaps"]},
    }


def validation_summary(val):
    if not val:
        return None
    return {
        "checks": len(val),
        "passed": sum(r["passed"] for r in val),
        "failed": [
            f"{r['case']} {r['quantity']} z={r['z']:+.2f} ({r['source']})"
            for r in val
            if not r["passed"]
        ],
    }


commit = subprocess.run(
    ["git", "log", "-1", "--format=%H"], capture_output=True, text=True
).stdout.strip()
r6 = load("gpu-r6.json")
summary = {
    "commit": commit,
    "device": r6["device"] if r6 else None,
    "validation_r5": validation_summary(load("gpu-r5-validation.json")),
    "validation_r6": validation_summary(load("gpu-r6-validation.json")),
    "cross_engine_r5": cross_summary(load("gpu-r5-cross-engine.json")),
    "cross_engine_r6": cross_summary(load("gpu-r6-cross-engine.json")),
    "benchmark": {
        "r1_first": bench_summary(load("gpu-r1-session.json")),
        "r3": bench_summary(load("gpu-r3-session.json")),
        "r5": bench_summary(load("gpu-r5.json")),
        "r6": bench_summary(r6),
        "r1_again": bench_summary(load("gpu-r1-again.json")),
    },
}
print("===== R5/R6 SUMMARY =====")
print(json.dumps(summary, indent=1))